In [204]:
import pandas as pd
import numpy as np

In [205]:
np.set_printoptions(suppress=True)

In [206]:
data = pd.read_csv('melb_data.csv')

In [207]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [208]:
data.isna().sum()

Suburb              0
Address             0
Rooms               0
Type                0
Price               0
Method              0
SellerG             0
Date                0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                62
Landsize            0
BuildingArea     6450
YearBuilt        5375
CouncilArea      1369
Lattitude           0
Longtitude          0
Regionname          0
Propertycount       0
dtype: int64

# I have to prepare a data set where price of the house will be predicted by :
- 1. Car Parking availability
- 2. Landsize
- 3. BuildingArea
- 4. YearBuilt.

In [209]:
# step 1 : Data preprocessing

In [210]:
newData = data[['Price','Car','Landsize','BuildingArea','YearBuilt']].copy()

In [211]:
newData.head()

,Price,Car,Landsize,BuildingArea,YearBuilt
0,1480000.0,1.0,202.0,NaN,NaN
1,1035000.0,0.0,156.0,79.0,1900.0
2,1465000.0,0.0,134.0,150.0,1900.0
3,850000.0,1.0,94.0,NaN,NaN
4,1600000.0,2.0,120.0,142.0,2014.0


In [212]:
# identify data columns
# Data columns (total 21 columns):s
#   Column         Non-Null Count  Dtype  
# ---  ------         --------------  -----  
#  4   Price          13580 non-null  float64  Continous (label) 
#  12  Car            13518 non-null  float64  discrete
#  13  Landsize       13580 non-null  float64  discrete
#  14  BuildingArea   7130 non-null   float64 Discrete
#  15  YearBuilt      8205 non-null   float64 DateTime(Year)
# dtypes: float64(12), int64(1), object(8)

In [213]:
#  15  YearBuilt      8205 non-null   float64 DateTime(Year)
# newData['YearBuilt'] = pd.to_datetime(newData['YearBuilt'],errors='coerce').dt.year

newData['YearBuilt'] = newData['YearBuilt'].fillna(int(newData['YearBuilt'].median()))


In [214]:
newData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         13580 non-null  float64
 1   Car           13518 non-null  float64
 2   Landsize      13580 non-null  float64
 3   BuildingArea  7130 non-null   float64
 4   YearBuilt     13580 non-null  float64
dtypes: float64(5)
memory usage: 530.6 KB


In [215]:
#  4   Price          13580 non-null  float64  Continous (label) 
from sklearn.impute import SimpleImputer

siForCar = SimpleImputer(missing_values=np.nan,strategy="mean")
siForCar.fit(newData[['Price']])
newData['Price'] = siForCar.transform(newData[['Price']])

In [216]:
#  13  Landsize       13580 non-null  float64  discrete

siForLandSize = SimpleImputer(missing_values=np.nan,strategy='median')
siForLandSize.fit(newData[['Landsize']])
newData['Landsize'] = siForLandSize.transform(newData[['Landsize']])


In [217]:
#  14  BuildingArea   7130 non-null   float64 Discrete

siForBuildingArea = SimpleImputer(missing_values=np.nan,strategy="median")
siForBuildingArea.fit(newData[['BuildingArea']])
newData['BuildingArea'] = siForBuildingArea.transform(newData[['BuildingArea']])

In [218]:
#  12  Car            13518 non-null  float64  discrete

siForCar = SimpleImputer(missing_values=np.nan,strategy="median")
siForCar.fit(newData[['Car']])
newData['Car'] = siForCar.transform(newData[['Car']])

In [219]:
newData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         13580 non-null  float64
 1   Car           13580 non-null  float64
 2   Landsize      13580 non-null  float64
 3   BuildingArea  13580 non-null  float64
 4   YearBuilt     13580 non-null  float64
dtypes: float64(5)
memory usage: 530.6 KB


In [220]:
features = newData.iloc[:,[1,2,3,4]].values
label = newData.iloc[:,0].values

In [221]:
features

array([[   1.,  202.,  126., 1970.],
       [   0.,  156.,   79., 1900.],
       [   0.,  134.,  150., 1900.],
       ...,
       [   4.,  436.,  126., 1997.],
       [   5.,  866.,  157., 1920.],
       [   1.,  362.,  112., 1920.]], shape=(13580, 4))

In [222]:
label

array([1480000., 1035000., 1465000., ..., 1170000., 2500000., 1285000.],
      shape=(13580,))

# step 2 : Correlation Analysis

In [223]:
# newData['YearBuilt'].head()
newData['YearBuilt'].describe()

count    13580.000000
mean      1966.788218
std         29.088642
min       1196.000000
25%       1960.000000
50%       1970.000000
75%       1975.000000
max       2018.000000
Name: YearBuilt, dtype: float64

In [224]:
newData.corr()

,Price,Car,Landsize,BuildingArea,YearBuilt
Price,1.000000,0.239109,0.037507,0.069763,-0.259387
Car,0.239109,1.000000,0.026780,0.068272,0.078696
Landsize,0.037507,0.026780,1.000000,0.094015,0.008806
BuildingArea,0.069763,0.068272,0.094015,1.000000,0.002359
YearBuilt,-0.259387,0.078696,0.008806,0.002359,1.000000


# step 3 : Feature elemination

In [225]:
allInFeature = np.append(np.ones((len(features),1)).astype(int),features,axis=1)

In [226]:
allInFeature

array([[   1.,    1.,  202.,  126., 1970.],
       [   1.,    0.,  156.,   79., 1900.],
       [   1.,    0.,  134.,  150., 1900.],
       ...,
       [   1.,    4.,  436.,  126., 1997.],
       [   1.,    5.,  866.,  157., 1920.],
       [   1.,    1.,  362.,  112., 1920.]], shape=(13580, 5))

In [227]:
from statsmodels.regression.linear_model import OLS

olsFormula = OLS(exog=allInFeature.astype(float),endog=label).fit()
olsFormula.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.139
Model:                            OLS   Adj. R-squared:                  0.138
Method:                 Least Squares   F-statistic:                     546.2
Date:                Fri, 01 May 2026   Prob (F-statistic):               0.00
Time:                        08:35:49   Log-Likelihood:            -1.9979e+05
No. Observations:               13580   AIC:                         3.996e+05
Df Residuals:                   13575   BIC:                         3.996e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.289e+07   3.45e+05     37.371      0.000    1.22e+07    1.36e+07
x1           1.71e+05   5330.370     32.076      0.000    1.61e+05    1.81e+05
x2             4.5451      1.282      3.545      0.000       2.032       7.058
x3            81.8466     13.070      6.262      0.000      56.227     107.467
x4         -6153.3334    175.621    -35.038      0.000   -6497.574   -5809.093
==============================================================================
Omnibus:                     6831.922   Durbin-Watson:                   1.468
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            77706.841
Skew:                           2.151   Prob(JB):                         0.00
Kurtosis:                      13.901   Cond. No.                     2.74e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.74e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

# step 4 : Model Building

- all features are selected

In [228]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
for rs in range(1,101):
    x_train,x_test,y_train,y_test = train_test_split(features,label,test_size=0.2,random_state=rs)
    
    model = LinearRegression()
    
    model.fit(x_train,y_train)
    
    trainScore = model.score(x_train,y_train)
    testScore = model.score(x_test,y_test)
    
    if testScore > trainScore and testScore >= 0.5 :
        print(f"Test Score {testScore} and Train Score {trainScore} , rs {rs}")
    # else : 
    #     print(f"Test Score {testScore} and Train Score {trainScore} , rs {rs} --------------> fail")


In [230]:
x_train,x_test,y_train,y_test = train_test_split(features,label,test_size=0.2,random_state=11)
model = LinearRegression()
    
model.fit(x_train,y_train)
    
trainScore = model.score(x_train,y_train)
testScore = model.score(x_test,y_test)

In [231]:
predictPrice = model.predict(np.array([[1,124,70,2000]]))

print(f"Predicted Price is {predictPrice}")

Predicted Price is [767240.94135787]
